# BoneNano step 5 — Paganin + FBP object initialization

Builds `obj_init.npy`, the warm start step 6 reads if `obj_init_file` is
uncommented in `config_step6.conf`. Ported FAITHFULLY from phascade's own
notebook for this dataset,
`phascade/experiments/bone_nano_holotomo/step2_scan-0086_catomo/1_catomo_recon.ipynb`
cell 30 — not from `AtomiumS2_coded`'s ablated settings, which are for a
different sample.

**Runs UNBINNED (bin 0), regardless of `config_step6.conf`'s current `bin`.**
`CodedReader.read_obj` always treats `obj_init.npy` as unbinned and bins it
down itself, the same rule it applies to `pdata`/`prb`/`code.npy`. So this
reads its own `config_steps15.conf`, not `config_step6.conf` — see that
file's header for why they are split.

Two differences from phascade's cell 30, both intentional:

- **No padding to `nzobj`/`nobj` before FBP.** phascade zero-pads the phase
  into the padded object canvas and shifts there with `ShiftFFT.curlyS`. Here
  the FBP volume stays `nz x n x n` (the detector size) and `CodedReader`'s
  own centring logic in `read_obj` places this smaller unbinned seed into the
  padded grid — the same choice `AtomiumS2_coded/step5_rec_paganin.ipynb`
  makes, and it is cheaper.
- **Analytic FBP, not his iterative `rec_tomo` (32 CG iterations).** Same
  speed/accuracy trade `AtomiumS2_coded` already made for an initial guess,
  with the same `FBP_SCALE` correction (see cell 6) — this is an initial
  guess for the solver, not the final reconstruction.

Everything else — the sub-pixel code window, the registration against
scan 86's own flat field, the eps and the GLOBAL (all-angle) background
subtraction — matches his notebook exactly.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('../../src'))
from holotomocupy.config import parse_args_coded_steps15
CFG = parse_args_coded_steps15('config_steps15.conf')
print(f'ntheta={CFG.ntheta}  n={CFG.n}  z1_code={CFG.z1_code}  rot_axis_offset={CFG.rot_axis_offset}')
print(f'paganin={CFG.paganin}  paganin_alpha={CFG.paganin_alpha}  '
      f'fbp_filter={CFG.fbp_filter!r}  mask_r={CFG.mask_r}')

## Geometry, positions, and the stage-0 warm start

In [ ]:
import numpy as np
import cupy as cp
import h5py
import matplotlib.pyplot as plt
%matplotlib inline
from holotomocupy.propagation import Propagation

PATH_H5  = CFG.in_file
PATH_OBJ = CFG.obj_init_file

with h5py.File(PATH_H5, 'r') as f:
    energy  = float(f['/exchange/energy'][0])
    det_px  = float(f['/exchange/detector_pixelsize'][0])   # UNBINNED
    fdd     = float(f['/exchange/focusdetectordistance'][0])
    z1      = float(f['/exchange/z1'][0])
    # ALL stored angles, same as AtomiumS2_coded/step5_rec_paganin.ipynb --
    # NOT a CFG.ntheta-sliced subset. step1/2/3 never subsample (see the
    # README), so this is the full 4001-angle scan, which is what a Paganin+
    # FBP seed wants anyway: more angles, not fewer. config_steps15.conf has
    # no start_theta/stride fields to replicate CodedReader's own read-time
    # subsampling, so slicing by CFG.ntheta here would silently take the
    # wrong frames (the first N, not step 6's chosen N-of-M) the moment
    # config_step6.conf's own stride stops being 1-of-1 -- taking everything
    # stored sidesteps that mismatch entirely.
    ntheta, nz, n = f['/exchange/pdata'].shape
    theta_deg   = f['/exchange/theta'][:].astype('float32')
    shifts_code = f['/exchange/shifts_code'][:].astype('float32')
    shifts_cor  = f['/exchange/shifts_cor'][:].astype('float32')
assert ntheta == CFG.ntheta, (
    f'{PATH_H5} has {ntheta} angles stored but config_steps15.conf says '
    f'ntheta={CFG.ntheta} -- update the config to match (informational only, '
    f'not used for slicing, but a mismatch means something upstream changed)')

z1_code = CFG.z1_code
# pos_obj = shifts_cor DIRECTLY. Step 3 already baked rot_axis_offset into
# shifts_cor before writing it (this folder's convention -- see the README's
# "Conventions that differ from stage 0"), unlike AtomiumS2 where step 3
# writes shifts_cor WITHOUT the offset and each later stage adds it
# independently. config_steps15.conf's rot_axis_offset is 0 for exactly this
# reason -- adding a nonzero value here would double-count it.
pos_obj = shifts_cor + np.asarray(CFG.rot_axis_offset, 'float32')

wavelength    = 1.24e-9 / energy
magnification = fdd / z1
distance      = z1 * (fdd - z1) / fdd
voxelsize     = det_px / magnification

z2c                     = z1 - z1_code
magnification_code      = z1 / z1_code
distance_code           = (z1_code * z2c) / (z1_code + z2c)
norm_magnification_code = magnification_code / (z1_code / z1)
distance_code           = distance_code * norm_magnification_code**2 * (z1_code / z1)**2

# UNBINNED stage-0 products, used AS-IS -- no bin2/crop_center needed the way
# step3 needs it at bin=3. code.npy is the full npsi x npsi plane; prb.npy
# already matches this stage's (nz, n) exactly.
code = np.load(CFG.code_file).astype('complex64')
prb  = np.load(CFG.prb_file ).astype('complex64')
ncode_full = code.shape[0]
assert prb.shape == (nz, n), f'{CFG.prb_file} is {prb.shape}, expected ({nz}, {n})'

cl_prop      = Propagation(n, nz, 1, 1, wavelength, voxelsize, [distance])
cl_prop_code = Propagation(n, nz, 1, 1, wavelength, voxelsize, [distance_code])

prb_gpu  = cp.asarray(prb)
code_gpu = cp.asarray(code)

print(f'ntheta={ntheta}  nz={nz}  n={n}  ncode_full={ncode_full}')
print(f'd1 (code->sample) {distance_code*1e3:+.4f} mm   d2 (sample->det) {distance*1e3:+.4f} mm')

## Per-angle flat field and Paganin helpers

In [ ]:
def fourier_shift_2d(arr, dc, dr):
    """Sub-pixel shift. Real in -> real out; complex in -> complex out."""
    nr, nc = arr.shape
    fr = cp.fft.fftfreq(nr).astype('float32')
    fc = cp.fft.fftfreq(nc).astype('float32')
    Fc, Fr = cp.meshgrid(fc, fr)
    ramp = cp.exp(-2j * cp.pi * (Fr * dr + Fc * dc)).astype('complex64')
    out = cp.fft.ifft2(cp.fft.fft2(arr.astype('complex64')) * ramp)
    return out if arr.dtype.kind == 'c' else out.real.astype('float32')

def code_window(k, G=8):
    """Code over the detector at angle k: integer crop from shifts_code,
    sub-pixel remainder by Fourier shift (G px guard band so nothing wraps in).
    Same precision as his ShiftFFT._sh_fwd -- NOT AtomiumS2's simpler
    integer-only crop."""
    dy, dx = int(round(shifts_code[k, 0])), int(round(shifts_code[k, 1]))
    cr, cc = ncode_full // 2 - dy, ncode_full // 2 - dx
    w = code_gpu[cr - nz // 2 - G: cr + nz // 2 + G, cc - n // 2 - G: cc + n // 2 + G]
    return fourier_shift_2d(w, dc=float(shifts_code[k, 1] - dx),
                            dr=float(shifts_code[k, 0] - dy))[G:-G, G:-G]

def flat_field(k):
    """What the detector records with no sample: the code window times the
    probe, propagated TOGETHER code plane -> sample -> detector -- the same
    forward-model piece RecCoded.F2 computes."""
    return cp.abs(cl_prop.D(cl_prop_code.D(prb_gpu * code_window(k), 0), 0)) ** 2

def multiPaganin(data, dist, wavelength, voxelsize, delta_beta, alpha):
    """Single-distance Paganin phase retrieval on a 2D CuPy array."""
    fy = cp.fft.fftfreq(data.shape[-2], d=voxelsize).astype('float32')
    fx = cp.fft.fftfreq(data.shape[-1], d=voxelsize).astype('float32')
    fx, fy    = cp.meshgrid(fx, fy)
    rad_freq  = cp.fft.fft2(data)
    taylorExp = 1 + wavelength * dist * cp.pi * delta_beta * (fx**2 + fy**2)
    numerator   = taylorExp * rad_freq
    denominator = taylorExp**2 + alpha
    phase = cp.log(cp.real(cp.fft.ifft2(numerator / denominator)))
    return delta_beta * 0.5 * phase

## Paganin phase, all angles

`eps=1e-3` is his value (`frames / (flat_ca + 1e-3)`), not AtomiumS2's
`1e-6`/`1e-5` -- this dataset's own settled choice. No Gaussian smoothing, no
edge crop: his bone_nano notebook uses neither.

In [ ]:
DELTA_BETA, ALPHA_PAG = CFG.paganin, CFG.paganin_alpha
EPS = 1e-3

CHUNK = 32
ph_all = np.empty((ntheta, nz, n), dtype='float32')

with h5py.File(PATH_H5, 'r') as f:
    for cs in range(0, ntheta, CHUNK):
        ce = min(cs + CHUNK, ntheta)
        frames = cp.asarray(f['/exchange/pdata'][cs:ce], dtype='float32')
        for j, k in enumerate(range(cs, ce)):
            proj = frames[j] / (flat_field(k) + EPS)
            ph_all[k] = multiPaganin(proj, distance, wavelength, voxelsize,
                                     DELTA_BETA, ALPHA_PAG).get()
        if cs % (CHUNK * 4) == 0:
            print(f'  {ce}/{ntheta}', flush=True)

# ONE background level over the WHOLE stack -- his cp.mean(ph[:, :n//16, :n//16]),
# computed over every angle together, not AtomiumS2's angle-0-only level.
corner = n // 16
global_bg = float(ph_all[:, :corner, :corner].mean())
ph_all -= global_bg
print(f'global_bg {global_bg:.6f} (mean of the {corner}x{corner} corner, all {ntheta} angles)')
print(f'ph_all [{ph_all.min():.4f}, {ph_all.max():.4f}]')

## Shift to centre the rotation axis

In [ ]:
rec_paganin = np.empty((ntheta, nz, n), dtype='float32')
for k in range(ntheta):
    phase = fourier_shift_2d(cp.asarray(ph_all[k]), dc=-pos_obj[k, 1], dr=-pos_obj[k, 0])
    rec_paganin[k] = phase.get()
del ph_all
print(f'rec_paganin {rec_paganin.shape}  [{rec_paganin.min():.4f}, {rec_paganin.max():.4f}]')

## FBP

In [ ]:
import threading
from holotomocupy.tomo import Tomo

theta_rad = np.deg2rad(theta_deg)
NGPU   = cp.cuda.runtime.getDeviceCount()
NCHUNK = 2

# FBP instead of his iterative rec_tomo (32 CG iterations): same speed trade
# AtomiumS2_coded already made for an initial guess.
#
# Tomo.fbp's norm_const leaves the result short by ~(pi/4)*sqrt(ntheta/n) --
# see AtomiumS2_coded/step5_rec_paganin.ipynb cell 4 for the derivation
# (verified over six geometries, n=256..2048, ntheta=375..7200). The
# correction depends only on ntheta/n, not on the sample, so it applies here
# unchanged.
FBP_SCALE  = (np.pi / 4) * np.sqrt(ntheta / n)
FBP_FILTER = CFG.fbp_filter

obj_fbp = np.zeros((nz, n, n), dtype='complex64')

def _worker(gpu_id, z_start, z_end):
    with cp.cuda.Device(gpu_id):
        cl = Tomo(n, NCHUNK, theta_rad, mask_r=CFG.mask_r)
        nz_loc = z_end - z_start
        for z0 in range(0, nz_loc, NCHUNK):
            z1_ = min(z0 + NCHUNK, nz_loc)
            sino = cp.array(rec_paganin[:, z_start + z0 : z_start + z1_, :], dtype='complex64')
            res  = cl.fbp(sino, FBP_FILTER) * FBP_SCALE
            obj_fbp[z_start + z0 : z_start + z1_] = res.real.astype('float32').get()
        print(f'GPU {gpu_id}: done  z={z_start}:{z_end}', flush=True)

chunk_z  = nz // NGPU
z_bounds = [(g * chunk_z, (g + 1) * chunk_z if g < NGPU - 1 else nz) for g in range(NGPU)]
threads  = [threading.Thread(target=_worker, args=(g, zs, ze)) for g, (zs, ze) in enumerate(z_bounds)]

print(f'FBP_SCALE = {FBP_SCALE:.4f}  (filter {FBP_FILTER!r}, ntheta={ntheta}, n={n}, {NGPU} GPUs)')
for t in threads: t.start()
for t in threads: t.join()

# obj.real = phase, the RecCoded variable itself; obj.imag = real / delta_beta,
# matching the exp(1j*k*R(u)) transmission model -- same convention steps15.py
# and AtomiumS2's notebook both use.
obj_fbp.imag = obj_fbp.real / DELTA_BETA

print(f'obj_fbp: {obj_fbp.shape}  real [{obj_fbp.real.min():.6f}, {obj_fbp.real.max():.6f}]  '
      f'imag [{obj_fbp.imag.min():.2e}, {obj_fbp.imag.max():.2e}]')

## Diagnostic: three orthogonal slices

In [ ]:
VIZ_Z = [nz // 4, nz // 2, 3 * nz // 4]

def _clim(sl):
    return float(np.percentile(sl, 1)), float(np.percentile(sl, 99))

fig, axes = plt.subplots(1, len(VIZ_Z), figsize=(6 * len(VIZ_Z), 6))
for col, z in enumerate(VIZ_Z):
    sl = obj_fbp[z].real
    vlo, vhi = _clim(sl)
    axes[col].imshow(sl, cmap='gray', vmin=vlo, vmax=vhi)
    axes[col].set_title(f'z = {z}', fontsize=13)
    axes[col].axis('off')
plt.suptitle('FBP -- phase (real part)', fontsize=14)
plt.tight_layout()
plt.show()

## Save

In [ ]:
np.save(PATH_OBJ, obj_fbp)
print(f'Saved to {PATH_OBJ}')
print('Uncomment obj_init_file in config_step6.conf to use it as the warm start.')